# F3 — Algoritmos para proporciones de ofertas
**Aporte de Víctor Bravo Barrera · Grupo 5**

Este notebook contiene el razonamiento, las pruebas, los resultados y la decisión de este aporte. El informe resume estos antecedentes. Se utilizan las clases de lectura, limpieza y validación existentes; sus módulos y la optimización de lectura asignada a Mauricio quedan fuera de esta intervención.

La unidad de observación sigue siendo una oferta por ítem. No se deduplican licitaciones ni se infiere causalidad.

**Recorrido:** entorno → entrada → limpieza → validación → contrato del cálculo → alternativas → pruebas → resultados analíticos → mediciones → complejidad → decisión.

Cada celda tiene un objetivo verificable. Las tablas de entrada, calidad, proporciones, tiempo y memoria se presentan por separado. Esta es una convención de organización del notebook: no significa que cada instrucción necesite su propia celda.

## 1. Entorno y presentación
### 1.1. Importaciones y ubicación del proyecto

In [ ]:
from pathlib import Path
import sys, json
import pandas as pd
raiz = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'src/analisis.py').exists())
if str(raiz) not in sys.path:
    sys.path.insert(0, str(raiz))
from src.entorno import versiones_entorno
from src.datos import ContratoEsquema, LectorCSV, sha256_archivo
from src.pipeline import LimpiadorLicitaciones
from src.validacion import ValidadorDatasetProcesado
from src.analisis import (tabla_proporciones, tabla_proporciones_iterativa,
                         tabla_proporciones_recursiva, tabla_proporciones_agrupada)

### 1.2. Funciones de presentación
Estas funciones cambian exclusivamente las etiquetas y el formato visible. Los DataFrames utilizados para pruebas y cálculos mantienen su estructura. Se definen aparte para que la configuración del entorno no mezcle lógica de presentación.

In [ ]:
# Formato de presentación: no modifica las tablas utilizadas en los cálculos.
NOMBRES_GRUPO = {'TamanoProveedor': 'Tamaño del proveedor', 'TipoLicitacion': 'Tipo de licitación'}
NOMBRES_ALGORITMO = {'referencia_f2': 'Referencia F2', 'iterativa': 'Iterativa',
                    'recursiva': 'Recursiva', 'agrupada': 'Agrupada'}

def mostrar_tabla(tabla, titulo, formatos=None):
    estilo = (tabla.style.hide(axis='index')
              .format(formatos or {}, na_rep='—', decimal=',', thousands='.', escape='html')
              .set_caption(titulo)
              .set_table_styles([
                  {'selector': 'caption', 'props': [('text-align', 'left'), ('font-weight', 'bold'), ('padding', '12px 0 6px')]},
                  {'selector': 'th', 'props': [('text-align', 'left'), ('padding', '8px 10px'), ('white-space', 'normal'), ('border-bottom', '2px solid #94a3b8')]},
                  {'selector': 'td', 'props': [('padding', '7px 10px'), ('text-align', 'right'), ('vertical-align', 'top'), ('font-variant-numeric', 'tabular-nums'), ('border-bottom', '1px solid #cbd5e1')]},
                  {'selector': 'td.col0', 'props': [('text-align', 'left'), ('max-width', '320px'), ('white-space', 'normal')]},
              ]))
    display(estilo)

def mostrar_proporciones(tabla, titulo):
    etiqueta = NOMBRES_GRUPO.get(tabla.index.name, tabla.index.name or 'Grupo')
    vista = tabla.rename(index={'All': 'Total'}).rename_axis(etiqueta).reset_index()
    vista = vista.rename(columns={'Ganadora': 'Ganadoras', 'Perdedora': 'Perdedoras',
        'Sin resultado válido': 'Sin resultado válido', 'All': 'Ofertas válidas',
        '% Ganadora': 'Ganadoras (%)', '% Perdedora': 'Perdedoras (%)'})
    mostrar_tabla(vista, titulo, {'Ganadoras': '{:,.0f}', 'Perdedoras': '{:,.0f}',
        'Sin resultado válido': '{:,.0f}', 'Ofertas válidas': '{:,.0f}',
        'Ganadoras (%)': '{:.2f}', 'Perdedoras (%)': '{:.2f}'})

### 1.3. Evidencia del entorno
Las versiones permiten interpretar y reproducir la ejecución. Los tiempos almacenados tienen además su propio registro de entorno.

In [3]:
print(sys.version)
mostrar_tabla(pd.DataFrame(list(versiones_entorno().items()), columns=['Dependencia', 'Versión']), 'Versiones del entorno')

3.12.14 (main, Aug 25 2026, 14:01:42) [MSC v.1944 64 bit (AMD64)]


Dependencia,Versión
numpy,2.3.5
pandas,3.0.1
jupyterlab,4.6.3
ipykernel,7.3.0
nbformat,5.11.1
nbconvert,7.17.1
nbclient,0.11.0


## 2. Obtención y preparación con las clases existentes
### 2.1. Entrada y contrato de lectura
`LectorCSV` hereda de `LectorDatos` y aplica `ContratoEsquema`. Se verifica la huella de la copia utilizada en F2. La lectura queda fuera del tiempo de los algoritmos.

In [4]:
ruta = raiz / 'data/raw/licitaciones_salud_marzo_2026.csv'
assert sha256_archivo(ruta) == '490d9209a10d387011d481b72b7891f26e997974ec2cf9dfc518aa4a08552232'
lector = LectorCSV(ContratoEsquema(('TamanoProveedor', 'TipoLicitacion', 'ResultadoOferta', 'EstadoLicitacion')))
raw = lector.leer(ruta)
mostrar_tabla(pd.DataFrame([{'Archivo': ruta.name, 'Filas': len(raw), 'Columnas': len(raw.columns)}]),
              'Dataset de entrada', {'Filas': '{:,.0f}', 'Columnas': '{:.0f}'})

Archivo,Filas,Columnas
licitaciones_salud_marzo_2026.csv,44.226,74


### 2.2. Limpieza y conservación de filas
El limpiador encapsula la configuración y un resumen de la última ejecución. Se reutiliza la lógica de F2. Esta celda muestra solamente la transformación estructural; las reglas de calidad aparecen después.

In [5]:
limpiador = LimpiadorLicitaciones()
datos = limpiador.limpiar(raw)
resumen = limpiador.ultima_ejecucion
mostrar_tabla(pd.DataFrame({
    'Etapa': ['Entrada', 'Salida'],
    'Filas': [resumen['filas_entrada'], resumen['filas_salida']],
    'Columnas': [resumen['columnas_entrada'], resumen['columnas_salida']],
}), 'Dimensiones antes y después de limpiar', {'Filas': '{:,.0f}', 'Columnas': '{:.0f}'})
print('Columnas excluidas:', ', '.join(resumen['columnas_excluidas']))

Etapa,Filas,Columnas
Entrada,44.226,74
Salida,44.226,74


Columnas excluidas: LicitacionBaseTipo, ContratoRenovable, UnidadTiempoRenovacion


Se conservan las 44.226 filas. La igualdad del número de columnas de entrada y salida no significa ausencia de transformaciones: se eliminan tres columnas vacías y se añaden tres variables derivadas.

### 2.3. Verificación de calidad
El validador ejecuta reglas mediante una interfaz común: ese intercambio de reglas evidencia polimorfismo. La siguiente tabla contiene exclusivamente las reglas superadas.

In [6]:
validacion = ValidadorDatasetProcesado().validar(datos, len(raw))
mostrar_tabla(pd.DataFrame({'Regla': validacion['detalle_reglas'], 'Estado': 'OK'}), 'Reglas de calidad superadas')

Regla,Estado
columnas_vacias_descartadas,OK
integridad_filas,OK
columnas_obligatorias,OK
resultado_oferta,OK
fechas,OK
variables_derivadas,OK


## 3. Contrato del cálculo y ejemplo manual
Se selecciona el estado `Adjudicada`. El denominador suma ganadoras y perdedoras por grupo; NA y resultados desconocidos se informan aparte. Un denominador cero produce NaN, mostrado como «—». Se rechazan grupos faltantes dentro del estado seleccionado y la categoría reservada `All`.

### 3.1. Datos del ejemplo
La oferta del grupo E está revocada y no participa del cálculo. El grupo B no tiene resultado válido.

In [7]:
from F3.test_algoritmos import muestra
mini = muestra()
mostrar_tabla(mini.rename(columns={'EstadoLicitacion': 'Estado', 'ResultadoOferta': 'Resultado'}), 'Ofertas del ejemplo manual')

Grupo,Estado,Resultado
A,Adjudicada,Ganadora
A,Adjudicada,Ganadora
A,Adjudicada,Perdedora
B,Adjudicada,—
C,Adjudicada,Perdedora
D,Adjudicada,Ganadora
E,Revocada,Ganadora


### 3.2. Resultado esperado
A tiene dos ganadoras de tres ofertas válidas: 66,67 %. C tiene 0 % y D tiene 100 %. El total es 3/5 = 60 %. Se suman los conteos; promediar porcentajes de grupos no produciría el mismo denominador.

In [8]:
resultado = tabla_proporciones_iterativa(mini, 'Grupo')
assert resultado.loc['All', '% Ganadora'] == 60
mostrar_proporciones(resultado, 'Proporciones del ejemplo')

Grupo,Ganadoras,Perdedoras,Sin resultado válido,Ofertas válidas,Ganadoras (%),Perdedoras (%)
A,2,1,0,3,"66,67","33,33"
B,0,0,1,0,—,—
C,0,1,0,1,"0,00","100,00"
D,1,0,0,1,"100,00","0,00"
Total,3,2,1,5,"60,00","40,00"


## 4. Alternativas y recursividad
### 4.1. Responsabilidades de cada alternativa

| Alternativa | Trabajo específico |
| --- | --- |
| Referencia F2 | Filtra resultados y agrupa varias veces; se conserva sin cambios. |
| Iterativa | Recorre registros y acumula tres contadores por grupo en un diccionario. |
| Recursiva | Divide intervalos, cuenta bloques y suma diccionarios parciales. |
| Agrupada | Agrupa una vez por categoría y código de resultado mediante pandas. |

Las tres alternativas nuevas comparten preparación y construcción de salida. La medición incluye esos pasos: no se excluyen conversiones que favorezcan a alguna alternativa. La agrupación utiliza `groupby` (The pandas development team, s. f.).

### 4.2. Descomposición recursiva
El caso base cuenta hasta 256 filas; un intervalo mayor se divide por el punto medio. Cada llamada reduce el rango y garantiza terminación. Se pasan posiciones, sin copiar mitades de DataFrames. La combinación suma conteos antes de calcular porcentajes.

La descomposición permite combinar resúmenes parciales, pero esta implementación es secuencial. El bloque 256 es configurable y no se presenta como óptimo.

In [9]:
import inspect
from src.analisis import _contar_dividiendo
print(inspect.getsource(_contar_dividiendo))

def _contar_dividiendo(grupos, codigos, inicio, fin, bloque_base):
    """Divide rangos, cuenta hojas y combina diccionarios en profundidad."""
    if fin - inicio <= bloque_base:
        return _contar_rango(grupos, codigos, inicio, fin)
    medio = (inicio + fin) // 2
    izquierda = _contar_dividiendo(grupos, codigos, inicio, medio, bloque_base)
    derecha = _contar_dividiendo(grupos, codigos, medio, fin, bloque_base)
    for grupo, valores in derecha.items():
        acumulado = izquierda.setdefault(grupo, [0, 0, 0])
        for i in range(3):
            acumulado[i] += valores[i]
    return izquierda



### 4.3. Comprobación de distintas particiones
Se compara el ejemplo con bloques de 1, 3 y 256. Los dos primeros fuerzan divisiones; el último cuenta directamente. La igualdad demuestra que la combinación conserva el resultado.

In [10]:
particiones = []
for bloque in (1, 3, 256):
    pd.testing.assert_frame_equal(tabla_proporciones_recursiva(mini, 'Grupo', bloque_base=bloque), resultado)
    particiones.append({'Bloque base': bloque, 'Equivalencia': 'OK'})
mostrar_tabla(pd.DataFrame(particiones), 'Verificación de particiones recursivas')

Bloque base,Equivalencia
1,OK
3,OK
256,OK


## 5. Pruebas y casos límite visibles
### 5.1. Suite reejecutable
Las ocho pruebas algorítmicas incluyen un oráculo manual, conservación de la entrada, índices repetidos, grupos inválidos, resultados desconocidos, vacíos, muestras aleatorias reproducibles y datos reales. Se ejecutan además las seis pruebas POO existentes. El registro detallado queda en esta salida.

In [11]:
import unittest, io
suite = unittest.defaultTestLoader.loadTestsFromNames(['F3.test_algoritmos', 'F3.test_nucleo_poo'])
log = io.StringIO()
pruebas = unittest.TextTestRunner(stream=log, verbosity=2).run(suite)
print(log.getvalue())
assert pruebas.wasSuccessful()

test_bloques_invalidos (F3.test_algoritmos.PruebasAlgoritmos.test_bloques_invalidos) ... ok
test_columnas_ausentes (F3.test_algoritmos.PruebasAlgoritmos.test_columnas_ausentes) ... ok
test_dataset_real_ambas_agrupaciones (F3.test_algoritmos.PruebasAlgoritmos.test_dataset_real_ambas_agrupaciones) ... ok
test_faltantes_resultados_e_indices_duplicados (F3.test_algoritmos.PruebasAlgoritmos.test_faltantes_resultados_e_indices_duplicados) ... ok
test_grupo_faltante_y_categoria_reservada (F3.test_algoritmos.PruebasAlgoritmos.test_grupo_faltante_y_categoria_reservada) ... ok
test_oraculo_manual_y_totales (F3.test_algoritmos.PruebasAlgoritmos.test_oraculo_manual_y_totales) ... ok
test_particiones_aleatorias_reproducibles (F3.test_algoritmos.PruebasAlgoritmos.test_particiones_aleatorias_reproducibles) ... ok
test_vacio_filtro_sin_filas_y_otro_estado (F3.test_algoritmos.PruebasAlgoritmos.test_vacio_filtro_sin_filas_y_otro_estado) ... ok
test_lector_aplica_contrato_y_adaptador_conserva_resultado (

### 5.2. Grupo sin resultados válidos
Se conserva el conteo de resultados sin clasificar, pero el denominador válido es cero. Mostrar 0 % sugeriría una proporción calculable que aquí no existe.

In [12]:
sin_resultado = mini.loc[mini.Grupo.eq('B')]
tabla_sin_resultado = tabla_proporciones_agrupada(sin_resultado, 'Grupo')
assert tabla_sin_resultado.loc['B', 'All'] == 0
assert pd.isna(tabla_sin_resultado.loc['B', '% Ganadora'])
mostrar_proporciones(tabla_sin_resultado, 'Caso límite: sin resultados válidos')

Grupo,Ganadoras,Perdedoras,Sin resultado válido,Ofertas válidas,Ganadoras (%),Perdedoras (%)
B,0,0,1,0,—,—
Total,0,0,1,0,—,—


### 5.3. Estado sin registros
Si el estado seleccionado no aparece, la salida contiene solamente el total cero y porcentajes no definidos. Es distinto del caso anterior, donde sí había un registro con resultado desconocido.

In [13]:
sin_filas = tabla_proporciones_agrupada(mini, 'Grupo', estado='Desierta')
assert len(sin_filas) == 1 and sin_filas.loc['All', 'All'] == 0
mostrar_proporciones(sin_filas, 'Caso límite: ningún registro seleccionado')

Grupo,Ganadoras,Perdedoras,Sin resultado válido,Ofertas válidas,Ganadoras (%),Perdedoras (%)
Total,0,0,0,0,—,—


### 5.4. Entrada inválida
Los faltantes de grupo dentro del estado seleccionado detienen el cálculo, evitando perder categorías silenciosamente. La excepción se captura aquí únicamente para mostrar la evidencia.

La referencia F2 usa acceso por atributo a `ResultadoOferta` y produce `AttributeError` si falta esa columna; las nuevas variantes usan acceso por columna y producen `KeyError`. La suite comprueba esa diferencia para entradas inválidas sin modificar F2.

In [14]:
entrada_invalida = mini.copy()
entrada_invalida.loc[0, 'Grupo'] = None
try:
    tabla_proporciones_agrupada(entrada_invalida, 'Grupo')
except ValueError as error:
    print('Error esperado:', error)
else:
    raise AssertionError('La entrada inválida no fue rechazada')

Error esperado: Grupo: resolver faltantes antes de agrupar.


## 6. Resultados del dataset real
### 6.1. Equivalencia de algoritmos
Antes de interpretar resultados o comparar tiempos, se verifica que cada alternativa conserve exactamente la tabla de referencia para ambas variables.

In [15]:
equivalencias = []
for grupo in ('TamanoProveedor', 'TipoLicitacion'):
    esperado = tabla_proporciones(datos, grupo)
    for nombre, funcion in [('Iterativa', tabla_proporciones_iterativa), ('Recursiva', tabla_proporciones_recursiva), ('Agrupada', tabla_proporciones_agrupada)]:
        pd.testing.assert_frame_equal(funcion(datos, grupo), esperado)
        equivalencias.append({'Variable': NOMBRES_GRUPO[grupo], 'Algoritmo': nombre, 'Resultado': 'Igual a F2'})
mostrar_tabla(pd.DataFrame(equivalencias), 'Equivalencia sobre el dataset completo')

Variable,Algoritmo,Resultado
Tamaño del proveedor,Iterativa,Igual a F2
Tamaño del proveedor,Recursiva,Igual a F2
Tamaño del proveedor,Agrupada,Igual a F2
Tipo de licitación,Iterativa,Igual a F2
Tipo de licitación,Recursiva,Igual a F2
Tipo de licitación,Agrupada,Igual a F2


### 6.2. Proporciones por tamaño de proveedor
Cada porcentaje utiliza las ofertas válidas de su propio grupo. Las diferencias son descriptivas y no demuestran un efecto causal del tamaño de empresa.

In [16]:
mostrar_proporciones(tabla_proporciones_agrupada(datos, 'TamanoProveedor'), 'Ofertas ganadoras por tamaño del proveedor')

Tamaño del proveedor,Ganadoras,Perdedoras,Sin resultado válido,Ofertas válidas,Ganadoras (%),Perdedoras (%)
Grande,12.575,6.255,0,18.830,"66,78","33,22"
Mediana,4.982,4.001,0,8.983,"55,46","44,54"
Micro,1.680,1.554,0,3.234,"51,95","48,05"
NoClasificado,1.803,1.576,0,3.379,"53,36","46,64"
Pequeña,5.022,4.577,0,9.599,"52,32","47,68"
Total,26.062,17.963,0,44.025,"59,20","40,80"


### 6.3. Proporciones por tipo de licitación
Se presenta en otra tabla porque corresponde a una agrupación distinta. Los denominadores deben acompañar a los porcentajes: categorías con pocas ofertas no tienen el mismo respaldo que las más numerosas.

In [17]:
mostrar_proporciones(tabla_proporciones_agrupada(datos, 'TipoLicitacion'), 'Ofertas ganadoras por tipo de licitación')

Tipo de licitación,Ganadoras,Perdedoras,Sin resultado válido,Ofertas válidas,Ganadoras (%),Perdedoras (%)
Licitación Privada Mayor a 1000 UTM,62,10,0,72,"86,11","13,89"
Licitación Privada Mayor a 5000 (I2),1,1,0,2,"50,00","50,00"
Licitación Privada entre 100 y 1000 UTM.,11,1,0,12,"91,67","8,33"
Licitación Pública Entre 100 y 1000 UTM (LE),14.686,8.904,0,23.590,"62,26","37,74"
Licitación Pública Mayor 1000 UTM (LP),8.066,6.328,0,14.394,"56,04","43,96"
Licitación Pública Mayor a 5000 (LR),1.652,1.559,0,3.211,"51,45","48,55"
Licitación Pública Menor a 100 UTM (L1),1.584,1.160,0,2.744,"57,73","42,27"
Total,26.062,17.963,0,44.025,"59,20","40,80"


## 7. Diseño experimental y trazabilidad
### 7.1. Qué se mide
Se comparan cuatro tamaños: 100, 1.000, 10.000 y 44.226 filas. Las muestras son anidadas y provienen de una permutación con semilla 2026. Cada alternativa recibe el mismo DataFrame.

Se realizan siete rondas de tres ejecuciones, con verificación y calentamiento previos y orden de algoritmos alternado con semilla. Se incluye filtro, conversión, conteo y tabla de salida. Se excluyen lectura, limpieza, muestreo y comprobación de equivalencia. La lectura corresponde al aporte de Mauricio.

`timeit` desactiva GC durante cada ronda. Se guarda el mínimo como referencia de menor interferencia, mediana, cuartiles y observaciones individuales. Los cuartiles describen la sesión; no son intervalos de confianza (Python Software Foundation, s. f.-a).

La memoria se mide tres veces en ejecuciones separadas con `tracemalloc`. Su pico excluye la entrada preexistente y no equivale al RSS ni garantiza incluir toda la memoria nativa (Python Software Foundation, s. f.-b).

### 7.2. Responsabilidades del script de medición

| Función | Responsabilidad |
| --- | --- |
| `medir` | Preparar tamaños y variables del experimento. |
| `medir_caso` | Comprobar equivalencia y coordinar un caso. |
| `medir_tiempos` | Reutilizar temporizadores y alternar las rondas. |
| `medir_memoria` | Recoger picos en ejecuciones separadas. |
| `resumir_medicion` | Calcular estadísticas y conservar observaciones. |
| `ejecutar` | Cargar mediante interfaces existentes y guardar evidencia. |

El conteo de filas adjudicadas se calcula una vez por muestra y los cuartiles en una operación. Los bucles restantes representan casos y repeticiones del experimento, no cruces entre todas las filas. Eliminarlos reduciría la evidencia; no demostraría una mejora algorítmica.

### 7.3. Evidencia utilizada
Por defecto se muestran mediciones guardadas cuyos hashes se comprueban. Para medir de nuevo, activar `REGENERAR_MEDICIONES`; si cambia el código, deben regenerarse. La nueva ejecución puede producir tiempos distintos.

In [18]:
from F3.medir_algoritmos import ejecutar
REGENERAR_MEDICIONES = False  # True repite el experimento y guarda nueva evidencia.
evidencia = raiz / 'evidencias/F3_algoritmos/mediciones.json'
if REGENERAR_MEDICIONES or not evidencia.exists():
    ejecutar()
registro = json.loads(evidencia.read_text(encoding='utf-8'))
assert registro['raw_sha256'] == sha256_archivo(ruta)
assert registro['analisis_sha256'] == sha256_archivo(raiz / 'src/analisis.py')
assert registro['medicion_sha256'] == sha256_archivo(raiz / 'F3/medir_algoritmos.py')
print('Mediciones de:', registro['fecha_utc'])
print('Python:', registro['python'])
print('Plataforma:', registro['sistema'])
print('Repeticiones:', registro['repeticiones'], '| Ejecuciones por ronda:', registro['ejecuciones_por_repeticion'])
mediciones = pd.DataFrame(registro['mediciones'])
assert len(mediciones) == 32
print('Casos experimentales:', len(mediciones))

Mediciones de: 2026-09-26T03:44:42.428446+00:00
Python: 3.12.14 (main, Aug 25 2026, 14:01:42) [MSC v.1944 64 bit (AMD64)]
Plataforma: Windows-11-10.0.26200-SP0
Repeticiones: 7 | Ejecuciones por ronda: 3
Casos experimentales: 32


## 8. Rendimiento temporal
### 8.1. Crecimiento por tamaño de proveedor
La tabla contiene solo medianas en milisegundos. Separar memoria y dispersión facilita comparar cómo cambia el tiempo al aumentar las filas.

In [19]:
def mostrar_crecimiento(grupo):
    vista = mediciones.loc[mediciones.grupo.eq(grupo)].pivot(index='filas_entrada', columns='algoritmo', values='mediana_ms')
    vista = vista.rename(columns=NOMBRES_ALGORITMO).rename_axis('Filas').reset_index()
    vista.columns.name = None
    mostrar_tabla(vista, NOMBRES_GRUPO[grupo] + ' — medianas (ms)',
                  {'Filas': '{:,.0f}', **{nombre: '{:.3f}' for nombre in NOMBRES_ALGORITMO.values()}})
mostrar_crecimiento('TamanoProveedor')

Filas,Agrupada,Iterativa,Recursiva,Referencia F2
100,"3,906","2,835","2,599","6,164"
1.000,"4,693","3,546","3,740","7,524"
10.000,"13,328","13,085","12,966","21,642"
44.226,"38,554","41,857","41,691","66,922"


### 8.2. Crecimiento por tipo de licitación
Es la misma operación sobre otra distribución de categorías. Se muestran los resultados por separado para evitar repetir una columna de agrupación en todas las filas.

In [20]:
mostrar_crecimiento('TipoLicitacion')

Filas,Agrupada,Iterativa,Recursiva,Referencia F2
100,"3,874","2,702","2,749","6,136"
1.000,"4,867","3,667","3,621","7,470"
10.000,"12,651","11,646","12,362","22,254"
44.226,"39,760","40,716","40,528","68,251"


### 8.3. Interpretación del crecimiento
En muestras pequeñas, agrupar tiene un costo fijo que puede superar al recorrido con diccionario. En 100 filas, el bloque base 256 evita que la variante recursiva divida: diferencias pequeñas frente a la iterativa no son evidencia de una ventaja de la recursividad.

Al aumentar n, pesa más el procesamiento de registros. La referencia realiza varios recorridos; las nuevas variantes comparten una preparación más directa. La recursiva no ofrece una ventaja sostenida sobre la iterativa en las mediciones guardadas. Ninguna tabla aislada demuestra una ley asintótica.

### 8.4. Dispersión temporal: tamaño de proveedor
Se muestran mínimo, mediana y cuartiles para las 44.226 filas, con 44.025 seleccionadas por estado. Son resúmenes de esta sesión, no garantías de rendimiento.

In [21]:
completo = mediciones.loc[mediciones.filas_entrada.eq(len(datos))].copy()
def mostrar_tiempos_completos(grupo):
    tabla = completo.loc[completo.grupo.eq(grupo)]
    vista = pd.DataFrame({'Algoritmo': tabla.algoritmo.map(NOMBRES_ALGORITMO),
                          'Mínimo (ms)': tabla.min_ms, 'Mediana (ms)': tabla.mediana_ms,
                          'Q25 (ms)': tabla.q25_ms, 'Q75 (ms)': tabla.q75_ms})
    mostrar_tabla(vista, NOMBRES_GRUPO[grupo] + ' — dispersión temporal',
                  {col: '{:.3f}' for col in vista.columns if col != 'Algoritmo'})
mostrar_tiempos_completos('TamanoProveedor')

Algoritmo,Mínimo (ms),Mediana (ms),Q25 (ms),Q75 (ms)
Referencia F2,"65,708","66,922","66,294","68,091"
Iterativa,"39,920","41,857","40,571","42,057"
Recursiva,"39,699","41,691","41,060","41,974"
Agrupada,"36,189","38,554","37,699","39,959"


### 8.5. Dispersión temporal: tipo de licitación
Los mínimos de las alternativas nuevas son cercanos. La ventaja de la agrupada por tipo debe interpretarse con cautela y junto con las otras métricas.

In [22]:
mostrar_tiempos_completos('TipoLicitacion')

Algoritmo,Mínimo (ms),Mediana (ms),Q25 (ms),Q75 (ms)
Referencia F2,"64,916","68,251","67,027","68,721"
Iterativa,"39,413","40,716","40,547","41,604"
Recursiva,"40,110","40,528","40,142","41,657"
Agrupada,"39,100","39,760","39,426","40,083"


## 9. Memoria auxiliar observada
La siguiente tabla contiene únicamente la mediana de los tres picos trazados para el conjunto completo, expresada en MiB (2²⁰ bytes). No mide la memoria total del CSV ni de todo el proceso.

In [23]:
memoria = completo.pivot(index='algoritmo', columns='grupo', values='pico_trazado_bytes_mediana') / 1024**2
memoria = memoria.rename(index=NOMBRES_ALGORITMO, columns=NOMBRES_GRUPO)
memoria.columns.name = None
memoria = memoria.rename_axis('Algoritmo').reset_index()
mostrar_tabla(memoria, 'Pico de memoria trazada — mediana (MiB)', {nombre: '{:.2f}' for nombre in NOMBRES_GRUPO.values()})

Algoritmo,Tamaño del proveedor,Tipo de licitación
Agrupada,"26,07","26,07"
Iterativa,"26,07","26,07"
Recursiva,"26,07","26,07"
Referencia F2,"40,81","40,81"


Las alternativas nuevas presentan picos muy similares entre sí y menores que la referencia. Esto sugiere que la preparación compartida pesa más que sus diferencias de conteo, pero no constituye una atribución causal del uso de memoria por función.

El filtrado conserva todas las columnas antes de extraer las necesarias. Reducir esa selección podría ser una mejora futura; no se presenta como trabajo ya realizado. La recursividad conserva diccionarios parciales, por lo que su costo espacial no se reduce a la pila de llamadas.

## 10. Complejidad temporal y espacial

Sea N el total de filas recibidas, n las seleccionadas, g los grupos y b el bloque base; el número de columnas se considera fijo. El filtro cuesta O(N), la preparación O(n), y ordenar categorías O(g log g).

| Alternativa | Conteo y combinación (costo esperado de hashing) | Memoria auxiliar del conteo |
| --- | --- | --- |
| Iterativa | O(n) | O(g) |
| Recursiva | O(n + S), donde S suma las entradas combinadas en nodos internos | Diccionarios parciales y pila; cota O(n + log L) |
| Agrupada | O(n + g), considerando tres resultados posibles | O(n + g) para estructuras de agrupación |
| Referencia F2 | Varios recorridos y agrupaciones; con tres resultados fijos, orden esperado lineal más ordenamiento | O(n + g) |

L = max(1, ceil(n/b)) aproxima la cantidad de hojas. La profundidad recursiva es O(log L); S está acotado por O(gL) y por O(n log L). Con g y b fijos, el conteo recursivo sigue siendo lineal; con muchos grupos diferentes puede acercarse a O(n log L). Los diccionarios de subárboles ya resueltos se conservan mientras se calcula la otra mitad: la memoria no se reduce a la pila.

Todas las funciones completas materializan datos filtrados y salida: la memoria auxiliar total es O(N + n + g), bajo columnas fijas, incluyendo la máscara. La recursividad no elimina esas asignaciones. Cambiar constantes puede mejorar los tiempos sin cambiar el orden asintótico.

## 11. Elección para F3 y alcance de la mejora
Se selecciona la variante agrupada para el análisis del conjunto completo: obtuvo la menor mediana en ambas variables y un pico trazado comparable con las otras alternativas nuevas. La referencia F2 permanece intacta por compatibilidad. La siguiente tabla calcula la razón entre tiempos; no representa una aceleración de toda la aplicación.

In [24]:
decisiones = []
for grupo in ('TamanoProveedor', 'TipoLicitacion'):
    tabla = completo.loc[completo.grupo.eq(grupo)].set_index('algoritmo')
    decisiones.append({'Variable': NOMBRES_GRUPO[grupo],
        'F2 (ms)': tabla.loc['referencia_f2', 'mediana_ms'],
        'Agrupada (ms)': tabla.loc['agrupada', 'mediana_ms'],
        'Razón F2 / agrupada': tabla.loc['referencia_f2', 'mediana_ms'] / tabla.loc['agrupada', 'mediana_ms']})
mostrar_tabla(pd.DataFrame(decisiones), 'Comparación que respalda la elección',
              {'F2 (ms)': '{:.3f}', 'Agrupada (ms)': '{:.3f}', 'Razón F2 / agrupada': '{:.2f}'})

Variable,F2 (ms),Agrupada (ms),Razón F2 / agrupada
Tamaño del proveedor,"66,922","38,554","1,74"
Tipo de licitación,"68,251","39,760","1,72"


La evidencia guardada muestra una razón cercana a 1,74 por tamaño y 1,72 por tipo. La ventaja frente a la referencia es mayor que la diferencia entre las tres alternativas nuevas. Por tipo, la diferencia respecto de la iterativa/recursiva es pequeña. Los datos no demuestran que pandas sea siempre más rápido ni justifican un selector automático según tamaño.

La recursividad demuestra descomposición, terminación y combinación correcta en un problema real. Se conserva como alternativa evaluada, aunque no resulte elegida para el análisis habitual. La selección se limita al volumen, cardinalidad, equipo y versiones registrados. Si se regeneran las mediciones, deben revisarse también estas conclusiones.

## 12. Reproducibilidad y cierre del aporte

Desde la raíz del repositorio:

```powershell
.\.venv\Scripts\python.exe -m unittest F3.test_algoritmos F3.test_nucleo_poo -v
.\.venv\Scripts\python.exe F3\medir_algoritmos.py
.\.venv\Scripts\python.exe F3\verificar_algoritmos.py
```

El script de medición conserva tiempos individuales, picos, parámetros, versiones y hashes en `evidencias/F3_algoritmos/`. El verificador ejecuta este notebook en un kernel nuevo y registra su hash. El informe `docs/F3_APORTE_ALGORITMOS.md` sintetiza las decisiones y resultados expuestos aquí; no contiene la única explicación de la selección.

Este aporte cubre algoritmos, recursividad, equivalencia y mediciones. La integración definitiva de Mauricio y Naya, las fuentes docentes/académicas y el informe institucional con revisión del PDF siguen siendo trabajo grupal pendiente.

## 13. Referencias técnicas
- Python Software Foundation. (s. f.-a). *timeit — Measure execution time of small code snippets*. https://docs.python.org/3/library/timeit.html
- Python Software Foundation. (s. f.-b). *tracemalloc — Trace memory allocations*. https://docs.python.org/3/library/tracemalloc.html
- The pandas development team. (s. f.). *pandas.DataFrame.groupby*. https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.groupby.html

Estas referencias respaldan el aporte técnico; la bibliografía grupal se completará durante la integración.